# QUELLE RÉPARTITION DES PRÊTS À LA MÉDIATHÈQUE ?

In [1]:
import pandas as pd
import numpy as np
from datetime import datetime as dt
import matplotlib.pyplot as plt
import matplotlib as mpl
import seaborn as sns
import kiblib.utils.lucas_sns_params
from kiblib.utils.db import DbConn

In [2]:
maj = dt.today().strftime('%d-%m-%Y à %Hh%M' )
print(f"Dernière mise à jour le {maj}")

Dernière mise à jour le 21-10-2026 à 15h29


In [3]:
db_conn = DbConn().create_engine()

In [4]:
query = """SELECT *
FROM koha_prod.items"""

In [5]:
items = pd.read_sql(query,db_conn)

In [6]:
cols2print = ["notforloan","damaged","damaged_on","barcode","location","ccode"]

In [7]:
items_damaged_2026 = items[(items["notforloan"]==4)&
                           (items["damaged"]==1)&
                           (items["damaged_on"]>"2026-01-01")&
                           (items["homebranch"]=="MED")
                          ][cols2print].sort_values(by="damaged_on")

In [8]:
indicateur = "nb_docs_SortisDesCollections_Abimés"

In [9]:
items_damaged_2026[f'{indicateur}'] = 1

In [10]:
location_fr = {"MED0A":"Collectivités",
               "MED0C":"Rez-de-chaussée",
               "MED1A":"Lac",
               "MED2A":"Jeunesse",
               "MED3A":"Phare",
               "MED3C":"Magasins - Diso 3e étage",
               "MED3D":"Magasins - Bibliothèque Professionnelle ",
               "MED3E":"Magasins - Pôle Adulte",
              }

In [11]:
location_fr.keys()

dict_keys(['MED0A', 'MED0C', 'MED1A', 'MED2A', 'MED3A', 'MED3C', 'MED3D', 'MED3E'])

In [12]:
items_damaged_2026.pivot_table(index="location",
                               values=indicateur,
                               aggfunc=sum
                              )

,nb_docs_SortisDesCollections_Abimés
location,
MED0A,3
MED0C,7
MED1A,467
MED2A,1052
MED3A,12
MED3C,172
MED3D,1
MED3E,1


In [13]:
for key,value in zip(location_fr.keys(),location_fr.values()):
    items_damaged_2026.loc[items_damaged_2026["location"]==key,["location"]] = value

In [14]:
stats_items_damaged = items_damaged_2026.pivot_table(index="location",
                               values=indicateur,
                               aggfunc=sum
                              )

In [15]:
stats_items_damaged.loc["Total"] = stats_items_damaged.sum()

In [16]:
for row in stats_items_damaged.index:
    stats_items_damaged.loc[row,"Part (En %)"] = round((stats_items_damaged.loc[row].sum() / stats_items_damaged.loc["Total"].sum())*100,2)

In [27]:
stats_items_damaged.sort_values("Part (En %)",ascending=False)

,nb_docs_SortisDesCollections_Abimés,Part (En %)
location,,
Total,1715,100.00
Jeunesse,1052,61.34
Lac,467,27.23
Magasins - Diso 3e étage,172,10.03
Phare,12,0.70
Rez-de-chaussée,7,0.41
Collectivités,3,0.17
Magasins - Bibliothèque Professionnelle,1,0.06
Magasins - Pôle Adulte,1,0.06
